# Домашняя работа 3

## Подготовка

In [1]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
print("pandas", pd.__version__, decisions.shape)

pandas 2.2.3 (6830, 7)


## Исходники домашней работы

In [2]:
org_keys = ["lei", "activity_year"]
long_keys = org_keys + ["action_taken"]
organizations = pd.read_csv(DATA / "organizations.csv", dtype={"lei": "string"})
lookup = organizations[org_keys + ["organization_name"]].copy()
assert not lookup.duplicated(org_keys).any()
print(organizations.head(3))

                    lei                       organization_name  source_count  \
0  549300AG64NHILB7ZP05                      LOANDEPOT.COM, LLC           114   
1  QFROUN1UWUYU0DVIWD51  Fifth Third Bank, National Association             3   
2  5493003GQDUH26DNNH17               NAVY FEDERAL CREDIT UNION           742   

   activity_year  
0           2023  
1           2023  
2           2023  


## H1 Профиль отказов

In [4]:
home = decisions.copy()
assert home[org_keys].notna().all().all()
assert home['action_taken'].isin([1, 2, 3]).all()
home['is_denied'] = home['action_taken'].eq(3)

denial_counts = home.groupby(org_keys, as_index=False).agg(
    n_decisions=('action_taken', 'size'),
    n_denied=('is_denied', 'sum')
)
denial_counts['denial_rate'] = denial_counts['n_denied'] / denial_counts['n_decisions']

denial = denial_counts.merge(lookup, on=org_keys, how='left',
                             validate='one_to_one', indicator=True)
assert denial['_merge'].eq('both').all()
assert len(denial) == len(denial_counts)
assert denial['n_decisions'].sum() == len(decisions)
assert denial['n_denied'].sum() == home['is_denied'].sum()
assert denial['denial_rate'].between(0, 1).all()
denial = denial.drop(columns='_merge')

denial.to_csv(WORK / 'home_denial_profile.csv', index=False)
print('Пример организации:')
print(denial.iloc[0].to_string())
denial.head()

Пример организации:
lei                           03D0JEWFDFUS0SEEKG89
activity_year                                 2023
n_decisions                                     91
n_denied                                        15
denial_rate                               0.164835
organization_name    TD Bank, National Association


,lei,activity_year,n_decisions,n_denied,denial_rate,organization_name
0,03D0JEWFDFUS0SEEKG89,2023,91,15,0.164835,"TD Bank, National Association"
1,1IE8VN30JCEQV1H4R804,2023,7,2,0.285714,"HSBC BANK USA, NATIONAL ASSOCIATION"
2,213800KVD76WKI6PN868,2023,2,1,0.500000,"CAPITAL CENTER, L.L.C."
3,213800QUAI2VH5YM6310,2023,89,9,0.101124,"EMBRACE HOME LOANS, INC."
4,213800XR2TCBQJSF1X93,2023,15,3,0.200000,NBKC BANK


## H2 Группы для просмотра

In [5]:
large_groups = denial.loc[denial['n_decisions'].ge(20)].copy()
large_groups.to_csv(WORK / 'home_denial_groups20.csv', index=False)

total_rate = denial['n_denied'].sum() / denial['n_decisions'].sum()
mean_rate = denial['denial_rate'].mean()

print('Всего групп:', len(denial))
print('Групп с 20 и более решениями:', len(large_groups))
print('Общая доля отказов, %:', round(total_rate * 100, 2))
print('Среднее долей по организациям, %:', round(mean_rate * 100, 2))
large_groups.head()

Всего групп: 314
Групп с 20 и более решениями: 61
Общая доля отказов, %: 10.12
Среднее долей по организациям, %: 14.22


,lei,activity_year,n_decisions,n_denied,denial_rate,organization_name
0,03D0JEWFDFUS0SEEKG89,2023,91,15,0.164835,"TD Bank, National Association"
3,213800QUAI2VH5YM6310,2023,89,9,0.101124,"EMBRACE HOME LOANS, INC."
9,2549003Z8R5QM0WPS603,2023,24,0,0.000000,IDB Global Federal Credit Union
14,2549006II76YXSS5XM65,2023,193,7,0.036269,"George Mason Mortgage, LLC"
26,254900FMBXF85WQYV433,2023,20,1,0.050000,Amalgamated Bank


## H3 Длинная, широкая и снова длинная форма

In [6]:
long_counts = decisions.groupby(long_keys).size().reset_index(name='n')

wide_counts = (long_counts.pivot(index=org_keys, columns='action_taken', values='n')
               .reindex(columns=[1, 2, 3]).fillna(0).astype('int64'))
wide_counts.columns.name = None

restored = wide_counts.reset_index().melt(
    id_vars=org_keys, value_vars=[1, 2, 3],
    var_name='action_taken', value_name='n'
)
restored['action_taken'] = restored['action_taken'].astype('int64')
restored = restored.loc[restored['n'].gt(0)].copy()

original = long_counts.sort_values(long_keys).reset_index(drop=True)
restored = restored.sort_values(long_keys).reset_index(drop=True)
assert_frame_equal(original, restored)
assert restored['n'].sum() == len(decisions)
print('Длинная форма:', original.shape)
print('Широкая форма:', wide_counts.shape)
print('Восстановленная форма:', restored.shape)
print('Проверка пройдена: счётчики совпадают')
restored.head()

Длинная форма: (500, 4)
Широкая форма: (314, 3)
Восстановленная форма: (500, 4)
Проверка пройдена: счётчики совпадают


,lei,activity_year,action_taken,n
0,03D0JEWFDFUS0SEEKG89,2023,1,76
1,03D0JEWFDFUS0SEEKG89,2023,3,15
2,1IE8VN30JCEQV1H4R804,2023,1,5
3,1IE8VN30JCEQV1H4R804,2023,3,2
4,213800KVD76WKI6PN868,2023,1,1


## H+ Безопасное соединение